#### Import Core Dependencies

In [67]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import GPT2TokenizerFast
from transformers import PreTrainedTokenizerFast
from typing import Any
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm.auto import tqdm
import ast
import statistics as stats
import re
from tokenizers import Tokenizer, trainers, pre_tokenizers, Regex
from tokenizers.models import BPE


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# verify paths are operational
print(Path("../data/raw/").exists())
print(Path("../checkpoints").exists())

True
True


In [4]:
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
CUDA Available: False


### Load and Inspect Data

In [5]:
# create path variables for ease of navigation
DATA_PATH = Path("../data/raw/")

lines_path = DATA_PATH / "movie_lines.txt"
conversation_path = DATA_PATH / "movie_conversations.txt"

In [6]:
# read first few lines
with open(lines_path, "r", encoding="iso-8859-1") as f:
    for _ in range(5):
        print(f.readline().strip())

L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!
L1044 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ They do to!
L985 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ I hope so.
L984 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ She okay?
L925 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Let's go.


In [7]:
with open(conversation_path, "r", encoding="iso-8859-1") as f:
    for _ in range(5):
        print(f.readline().strip())

u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L198', 'L199']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L200', 'L201', 'L202', 'L203']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L204', 'L205', 'L206']
u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L207', 'L208']


In [8]:
with open(lines_path, "r", encoding="iso-8859-1") as f:
    num_lines = sum(1 for _ in f)

print(f"Movie lines: {num_lines:,}")

Movie lines: 304,713


In [9]:
with open(conversation_path, "r", encoding="iso-8859-1") as f:
    num_conversations = sum(1 for _ in f)

print(f"Conversations: {num_conversations:,}")
# the documentation mentioned over 220K conversational exchanges which I presume would not whole conversations which is counted below

Conversations: 83,097


In [10]:
with open(
    lines_path,
    encoding="iso-8859-1"
) as f:

    sample_line = f.readline()

print(sample_line)
print(sample_line.split(" +++$+++ "))

L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!

['L1045', 'u0', 'm0', 'BIANCA', 'They do not!\n']


In [11]:
with open(
    conversation_path,
    encoding="iso-8859-1"
) as f:

    sample_conv = f.readline()

print(sample_conv)
print(sample_conv.split(" +++$+++ "))

u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']

['u0', 'u2', 'm0', "['L194', 'L195', 'L196', 'L197']\n"]


##### Initial Dataset Inspection

Initial inspection of the Cornell Movie-Dialogs Corpus revealed that the dataset is stored in a relational format rather than as fully assembled conversations. The `movie_lines.txt` file contains individual utterances associated with line identifiers, characters, and movies, while `movie_conversations.txt` contains ordered collections of line identifiers representing conversations between characters. Exploratory analysis identified approximately 304,713 individual movie lines and 83,097 conversations.

To better understand the conversational content and prepare the data for language modeling, the next step is to reconstruct complete conversations by mapping line identifiers to their corresponding utterances. This reconstruction process will produce a coherent dialogue corpus suitable for further exploratory analysis, tokenization, and Transformer training.

In [12]:
line_lookup = {}

with open(lines_path,"r", encoding="iso-8859-1") as f:

    for line in f:

        parts = line.strip().split(" +++$+++ ")

        line_id = parts[0]

        text = parts[-1]

        line_lookup[line_id] = text

print(
    f"Lines Loaded: {len(line_lookup):,}"
)

Lines Loaded: 304,713


In [13]:
sample_key = next(iter(line_lookup))

print(sample_key)

print(
    line_lookup[sample_key]
)

L1045
They do not!


In [14]:
conversations = []

with open(conversation_path, "r", encoding="iso-8859-1") as f:
    for line in f:
        parts = line.strip().split(" +++$+++ ")
        line_ids = ast.literal_eval(parts[-1])
        
        # Builds the conversation
        dialogue = [line_lookup[line_id] for line_id in line_ids if line_id in line_lookup]
        
        if dialogue:
            conversations.append(dialogue)


print(
    f"Conversations Reconstructed: {len(conversations):,}"
)

Conversations Reconstructed: 83,097


In [15]:
sample_conversation = conversations[2]

for i, utterance in enumerate(
    sample_conversation,
    start=1
):
    print(
        f"{i}. {utterance}"
    )

1. No, no, it's my fault -- we didn't have a proper introduction ---
2. Cameron.
3. The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does.
4. Seems like she could get a date easy enough...


In [16]:
for i in range(3):

    print(f"\nConversation {i+1}")
    print("-" * 40)

    for utterance in conversations[i]:
        print(utterance)



Conversation 1
----------------------------------------
Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.
Well, I thought we'd start with pronunciation, if that's okay with you.
Not the hacking and gagging and spitting part.  Please.
Okay... then how 'bout we try out some French cuisine.  Saturday?  Night?

Conversation 2
----------------------------------------
You're asking me out.  That's so cute. What's your name again?
Forget it.

Conversation 3
----------------------------------------
No, no, it's my fault -- we didn't have a proper introduction ---
Cameron.
The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does.
Seems like she could get a date easy enough...


##### Intial Data Preparation

- We will be adding boundary markers to our corpus so the Transformer model can learn turn behaviour

In [17]:
EOU_TOKEN = "<|eou|>"
EOD_TOKEN = "<|endofdialog|>"

In [18]:
processed_conversations = []

for conversation in conversations:

    dialogue_text = (
        f" {EOU_TOKEN} "
    ).join(conversation)

    dialogue_text += f" {EOD_TOKEN}"

    processed_conversations.append(
        dialogue_text
    )

In [19]:
processed_conversations[0:3]

["Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again. <|eou|> Well, I thought we'd start with pronunciation, if that's okay with you. <|eou|> Not the hacking and gagging and spitting part.  Please. <|eou|> Okay... then how 'bout we try out some French cuisine.  Saturday?  Night? <|endofdialog|>",
 "You're asking me out.  That's so cute. What's your name again? <|eou|> Forget it. <|endofdialog|>",
 "No, no, it's my fault -- we didn't have a proper introduction --- <|eou|> Cameron. <|eou|> The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does. <|eou|> Seems like she could get a date easy enough... <|endofdialog|>"]

##### Statistical Inspection after Initial Data Preparation

In [32]:
# statistics for dataset
word_counts = [len(conv.lower().split()) for conv in processed_conversations]


print(f"Conversations : {len(processed_conversations):,}")
print(f"Total Words : {sum(word_counts):,}")

print(f"Min Words : {min(word_counts)}")
print(f"Mean Words : {stats.mean(word_counts):.2f}")
print(f"Median Words : {stats.median(word_counts)}")
print(f"Max Words : {max(word_counts)}")

Conversations : 83,097
Total Words : 3,515,653
Min Words : 4
Mean Words : 42.31
Median Words : 28
Max Words : 1245


In [39]:
# we want true unique words that strips punctuation so we can understand the vocabulary size
all_words_re = []

for conversation in processed_conversations:

    words = re.findall(r"\b[\w']+\b", conversation.lower())

    all_words_re.extend(words)

vocabulary_re = set(all_words_re)

print(
    f"Total Words: {len(all_words_re):,}"
)

print(
    f"Unique Words with stripping punctuation: {len(vocabulary_re):,}"
)

Total Words: 3,515,026
Unique Words with stripping punctuation: 53,345


In [42]:
all_words = []

for conversation in processed_conversations:

    all_words.extend(
        conversation.lower().split()
    )

vocabulary = set(all_words)

print(
    f"Total Words: {len(all_words):,}"
)

print(
    f"Unique Words by keeping punctuation: {len(vocabulary):,}"
)

Total Words: 3,515,653
Unique Words by keeping punctuation: 143,207


In [ ]:
word_freq = Counter(all_words_re)
word_freq.most_common(25)
# looking for boundaries and pronouns

[('eou', 221616),
 ('you', 128678),
 ('i', 104713),
 ('the', 99321),
 ('endofdialog', 83097),
 ('to', 80764),
 ('a', 71526),
 ('it', 47496),
 ('and', 46147),
 ('of', 39480),
 ('that', 35450),
 ('in', 34387),
 ('what', 32322),
 ('me', 31906),
 ('is', 29284),
 ('this', 24595),
 ("don't", 24349),
 ('for', 23447),
 ('do', 22788),
 ("i'm", 22524),
 ('know', 21686),
 ('have', 21529),
 ('your', 20972),
 ('my', 20861),
 ('we', 20748)]

In [41]:
rare_words = [word for word, count in word_freq.items() if count == 1]
print(
    f"Words appearing once: {len(rare_words):,} ({(len(rare_words) / len(vocabulary_re)) * 100:.1f}% of vocab)"
)

print(rare_words[:50])

Words appearing once: 20,920 (39.2% of vocab)
['roxanne', 'korrine', 'pronunciation', "c'esc", 'tete', "eber's", 'blowdryer', 'diffuser', 'guillermo', '90210', 'bonchowski', "lowenbrau's", 'clouted', 'fen', "mandella's", 'obsess', "bogey's", "kat's", 'brucie', 'padua', 'undersexed', 'grandmothers', 'diluted', 'perma', 'consecrate', 'shrew', 'mewling', 'rampalian', 'thrives', 'alcatraz', 'orchestrating', 'assail', 'buttholus', 'extremus', 'kmart', 'mandella', 'backfired', 'burnside', 'screwboy', 'fenders', 'mainline', 'mystique', 'fallacy', "scurvy's", 'strat', 'veggie', 'allure', 'blaise', "rictor's", 'gonad']


In [21]:
character_counts = [len(conv) for conv in processed_conversations] # each conv is a string now so we count each character like this

print(f"Mean Characters : {stats.mean(character_counts):.2f}")
print(f"Median Characters : {stats.median(character_counts)}")
print(f"Max Characters : {max(character_counts)}")

Mean Characters : 242.72
Median Characters : 167
Max Characters : 6678


##### Summary of Initial Data Preparation and Statistical Inspection

The Cornell Movie-Dialogs Corpus stores conversations indirectly through references to individual dialogue lines. To create a language modeling corpus suitable for Transformer training, conversations were reconstructed by mapping the ordered line identifiers contained in `movie_conversations.txt` to their corresponding utterances in `movie_lines.txt`. This process transformed the original relational dataset into complete conversational exchanges while preserving dialogue order.

To explicitly represent conversational structure, two custom boundary markers were introduced. The `<|eou|>` token was inserted between utterances to indicate the end of an individual speaker turn, while `<|endofdialog|>` was appended to the end of each conversation to mark dialogue termination. These markers provide the model with explicit signals regarding conversational boundaries and turn-taking behavior, reducing the risk of learning artificial transitions between unrelated conversations.

The reconstructed corpus contains 83,097 complete conversations comprising approximately 3.5 million words. Conversations are generally short, with an average length of 42.31 words and a median length of 28 words, although a small number of substantially longer conversations are present. These statistics suggest a dialogue corpus dominated by concise conversational exchanges while still providing sufficient variability for language model training.

**Notes on Unique Word Count and Frequency**

A preliminary vocabulary analysis showed a substantial difference between raw whitespace tokenization and normalized lexical analysis. The reconstructed corpus contained approximately 53,345 unique lexical words after normalizing case and punctuation. In contrast, simple whitespace splitting produced over 143,207 unique tokens because punctuation variants, formatting differences, and special symbols were treated as distinct vocabulary items. Out of the 53,345 unique words, 20,920 words appeared only once, representing approximately 39% of the vocabulary. Many rare words corresponded to character names, locations, proper nouns, and movie-specific terminology, refelcting a longer tailed distribution of words. These observations further motivate the use of **subword tokenization techniques such as GPT-2 Byte Pair Encoding**, which are designed to manage large and inconsistent vocabularies more effectively than word-level representations.

##### Tokenization
 - We will build on the previous project and test a custom tokenizer to see if it will have an optimization effect on our training

In [50]:
# Does the pre built tokenizer split out boundary tags? highly likely
tokenizer = GPT2TokenizerFast.from_pretrained(
    "gpt2"
)

print(
    tokenizer.tokenize(
        processed_conversations[0]
    )[:50]
)

['Can', 'Ġwe', 'Ġmake', 'Ġthis', 'Ġquick', '?', 'Ġ', 'ĠRox', 'anne', 'ĠKor', 'rine', 'Ġand', 'ĠAndrew', 'ĠBarrett', 'Ġare', 'Ġhaving', 'Ġan', 'Ġincredibly', 'Ġhorrendous', 'Ġpublic', 'Ġbreak', '-', 'Ġup', 'Ġon', 'Ġthe', 'Ġquad', '.', 'Ġ', 'ĠAgain', '.', 'Ġ<', '|', 'e', 'ou', '|', '>', 'ĠWell', ',', 'ĠI', 'Ġthought', 'Ġwe', "'d", 'Ġstart', 'Ġwith', 'Ġpronunciation', ',', 'Ġif', 'Ġthat', "'s", 'Ġokay']


In [ ]:
# register our special tokens so it can be identified
special_tokens = {
    "additional_special_tokens": [
        "<|eou|>",
        "<|endofdialog|>"
    ]
}

num_added = tokenizer.add_special_tokens(
    special_tokens
)

print(
    f"Added Special Tokens: {num_added}"
)

print(
    f"Tokenizer Vocabulary Size: {len(tokenizer)}"
)

Added Special Tokens: 2
Tokenizer Vocabulary Size: 50259


In [52]:
# Test if tokenizer now recognizes boundary tokens
example = ("Hello there! " "<|eou|> " "How are you? " "<|endofdialog|>"
)

print(tokenizer.tokenize(example))

['Hello', 'Ġthere', '!', 'Ġ', '<|eou|>', 'ĠHow', 'Ġare', 'Ġyou', '?', 'Ġ', '<|endofdialog|>']


In [53]:
token_lengths = []

total_tokens = 0

for conversation in processed_conversations:

    token_ids = tokenizer.encode(
        conversation,
        add_special_tokens=False
    )

    token_lengths.append(
        len(token_ids)
    )

    total_tokens += len(token_ids)

print(
    f"Total Tokens: {total_tokens:,}"
)

print(
    f"Mean Tokens: "
    f"{stats.mean(token_lengths):.2f}"
)

print(
    f"Median Tokens: "
    f"{stats.median(token_lengths)}"
)

print(
    f"Max Tokens: "
    f"{max(token_lengths)}"
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1146 > 1024). Running this sequence through the model will result in indexing errors


Total Tokens: 5,085,643
Mean Tokens: 61.20
Median Tokens: 42
Max Tokens: 1682


In [54]:
used_tokens = set()

for conversation in processed_conversations:

    token_ids = tokenizer.encode(
        conversation,
        add_special_tokens=False
    )

    used_tokens.update(
        token_ids
    )

vocab_size = len(tokenizer)

print(
    f"Used Token IDs: "
    f"{len(used_tokens):,}"
)

print(
    f"Vocabulary Size: "
    f"{vocab_size:,}"
)

print(
    f"Vocabulary Utilization: "
    f"{len(used_tokens)/vocab_size:.2%}"
)

Used Token IDs: 36,879
Vocabulary Size: 50,259
Vocabulary Utilization: 73.38%


In [ ]:
VOCAB_SIZE = 16_384
TRAIN_TOKENIZER = False

# We want to avoid retraining the tokenizer so we save and load it later

if TRAIN_TOKENIZER:

    # Train custom tokenizer for our corpus
    tokenizer_obj = Tokenizer(
        BPE(unk_token="<|unk|>")
    )

    tokenizer_obj.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(
            pattern=Regex("<\|unk\|>|<\|eou\|>|<\|endofdialog\|>"),
            behavior="isolated"
        ),
        pre_tokenizers.WhitespaceSplit(),
        pre_tokenizers.ByteLevel(add_prefix_space=False)
    ])



    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=[
            "<|unk|>",
            "<|eou|>",
            "<|endofdialog|>"
        ]
    )

    tokenizer_obj.train_from_iterator(
        processed_conversations,
        trainer=trainer
    )


In [ ]:
if TRAIN_TOKENIZER:

    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_obj,

        unk_token="<|unk|>",

        eos_token="<|endofdialog|>",

        additional_special_tokens=[
            "<|eou|>"
        ]
    )

In [ ]:
if TRAIN_TOKENIZER:

    TOKENIZER_DIR = Path("../tokenizers/cornell_bpe")

    hf_tokenizer.save_pretrained(TOKENIZER_DIR)

    print(f"Tokenizer saved to: {TOKENIZER_DIR}")

Tokenizer saved to: ..\tokenizers\cornell_bpe


In [74]:
hf_tokenizer = (
    PreTrainedTokenizerFast
    .from_pretrained(
        TOKENIZER_DIR
    )
)

In [75]:
print(
    hf_tokenizer.vocab_size
)

print(
    hf_tokenizer.special_tokens_map
)

16384
{'eos_token': '<|endofdialog|>', 'unk_token': '<|unk|>'}


In [78]:
sample = ("Hello there! " "<|eou|> " "How are you? " "<|endofdialog|>")

encoding_ids = hf_tokenizer.encode(sample, add_special_tokens=False)

# 3. Get the string tokens (to visually verify your boundaries)
tokens = hf_tokenizer.tokenize(sample, add_special_tokens=False)

# Test if successful
print(f"Encoded IDs: {encoding_ids}")
print(f"Tokens:      {tokens}")

Encoded IDs: [1065, 289, 3, 1, 356, 230, 144, 32, 2]
Tokens:      ['Hello', 'there', '!', '<|eou|>', 'How', 'are', 'you', '?', '<|endofdialog|>']


In [79]:
encoding_ids = hf_tokenizer.encode(
    processed_conversations[0],
    add_special_tokens=False
)

tokens = hf_tokenizer.tokenize(
    processed_conversations[0],
    add_special_tokens=False
)

print(f"Token Count: {len(encoding_ids)}")

print(tokens[:60])


Token Count: 87
['Can', 'we', 'make', 'this', 'quick', '?', 'Ro', 'x', 'anne', 'Kor', 'rine', 'and', 'Andrew', 'Barrett', 'are', 'having', 'an', 'incredibly', 'hor', 'rend', 'ous', 'public', 'break', '-', 'up', 'on', 'the', 'quad', '.', 'Again', '.', '<|eou|>', 'Well', ',', 'I', 'thought', 'we', "'d", 'start', 'with', 'pron', 'un', 'ci', 'ation', ',', 'if', 'that', "'s", 'okay', 'with', 'you', '.', '<|eou|>', 'Not', 'the', 'hack', 'ing', 'and', 'gag', 'ging']


In [80]:
custom_token_lengths = []
custom_used_tokens = set()

for conversation in processed_conversations:
    # Direct list output (removed .ids) + ensured add_special_tokens=False
    token_ids = hf_tokenizer.encode(
        conversation,
        add_special_tokens=False
    )

    custom_token_lengths.append(
        len(token_ids)
    )

    # Update set directly with the integer list
    custom_used_tokens.update(
        token_ids
    )

# Use len() on the hf_tokenizer to accurately count the total vocabulary
vocab_size = len(hf_tokenizer)

print(
    f"Used Token IDs: "
    f"{len(custom_used_tokens):,}"
)

print(
    f"Vocabulary Size: "
    f"{vocab_size:,}"
)

print(
    f"Vocabulary Utilization: "
    f"{len(custom_used_tokens)/vocab_size:.2%}"
)


Used Token IDs: 16,022
Vocabulary Size: 16,384
Vocabulary Utilization: 97.79%


In [81]:
total_tokens = sum(custom_token_lengths)

print(
    f"Total Tokens: {total_tokens:,}"
)

print(
    f"Mean Tokens: "
    f"{stats.mean(custom_token_lengths):.2f}"
)

print(
    f"Median Tokens: "
    f"{stats.median(custom_token_lengths)}"
)

print(
    f"Max Tokens: "
    f"{max(custom_token_lengths)}"
)

Total Tokens: 4,694,594
Mean Tokens: 56.50
Median Tokens: 38
Max Tokens: 1609


##### Tokenizer Experiment Summary

Following corpus reconstruction and vocabulary analysis, two tokenization approaches were evaluated for the corpus. The first approach used the pretrained GPT-2 Byte Pair Encoding (BPE) tokenizer with the custom dialogue boundary tokens <|eou|> and <|endofdialog|> added as special tokens. The second approach trained a domain-specific BPE tokenizer directly on the reconstructed movie dialogue corpus using a vocabulary size of 16,384 tokens and explicitly incorporated the dialogue boundary markers as special tokens during training.

The GPT-2 tokenizer produced a corpus containing 5,085,643 tokens with an average conversation length of 61.20 tokens and achieved 73.38% vocabulary utilization. Although GPT-2 performed substantially better on this corpus than it did in the previous DailyDialog project, inspection revealed that its vocabulary remained considerably larger than necessary for the domain-specific movie dialogue task.

The custom tokenizer reduced the corpus size to 4,694,594 tokens, representing a reduction of approximately 391,000 tokens (7.7%) compared to GPT-2. Average conversation length decreased from 61.20 to 56.50 tokens, while vocabulary utilization increased to 97.79%, indicating that nearly the entire tokenizer vocabulary was actively used by the corpus. Additionally, the custom tokenizer preserved the dialogue structure tokens <|eou|> and <|endofdialog|> as atomic symbols and learned meaningful movie-dialogue-specific subword representations.

Based on these results, the custom tokenizer was selected for the remainder of the project. It provides better domain alignment, improved corpus compression, more efficient vocabulary utilization, and explicit support for the conversational structure introduced during preprocessing. These advantages are expected to improve training efficiency while preserving the linguistic characteristics of the movie dialogue corpus.